# General tensor-MD pipeline

This notebook demonstrates the package without MVTec-specific folders or labels:

`normal images → feature extractor → tensor patches → fit → score new images → diagnostics`

The example uses a small stand-in feature extractor so it runs without downloading a CNN. Replace it with `make_cnn_feature_extractor(your_model, ...)` for a real Keras or PyTorch model.

In [ ]:
from pathlib import Path
import tempfile

import numpy as np
from PIL import Image

from tensor_md import (
    LocationAwareTensorMahalanobisDetector,
    PatchExtractionConfig,
    load_image_patches,
    load_normal_patches,
)

In [ ]:
# Create two ordinary folders: one for normal fitting images and one for images to inspect.
root = Path(tempfile.mkdtemp(prefix="tensor-md-demo-"))
normal_dir = root / "normal"
query_dir = root / "to_check"
normal_dir.mkdir()
query_dir.mkdir()

rng = np.random.default_rng(4)
for index in range(6):
    image = np.full((32, 32, 3), 0.35, dtype=np.float32)
    image += rng.normal(0, 0.025, image.shape).astype(np.float32)
    Image.fromarray(np.clip(image * 255, 0, 255).astype(np.uint8)).save(normal_dir / f"normal_{index}.png")

# One query image contains a bright local change.
query = np.full((32, 32, 3), 0.35, dtype=np.float32)
query[12:18, 20:26] += 0.5
Image.fromarray(np.clip(query * 255, 0, 255).astype(np.uint8)).save(query_dir / "query_0.png")
print(root)

## Supply any feature extractor

The package expects a callable that receives an NHWC float batch in `[0, 1]` and returns one or more NHWC feature-map batches. This stand-in keeps two channels and downsamples the images; a real CNN can be supplied instead.

In [ ]:
def feature_extractor(images):
    # Replace this function with make_cnn_feature_extractor(your_model, ...).
    pooled = images[:, ::2, ::2, :]
    brightness = pooled.mean(axis=-1, keepdims=True)
    contrast = pooled.max(axis=-1, keepdims=True) - pooled.min(axis=-1, keepdims=True)
    return np.concatenate([brightness, contrast], axis=-1).astype(np.float32)

In [ ]:
config = PatchExtractionConfig(
    train_image_dir=normal_dir,
    image_size=(32, 32),
    input_representation="cnn_features",
    cnn_backbone="custom",
    cnn_feature_extractor=feature_extractor,
    cnn_feature_patch_size=(1, 1),
)

normal = load_normal_patches(config)
to_check = load_image_patches(query_dir, config)
print("normal patches:", normal.patches.shape)
print("query patches:", to_check.patches.shape)

In [ ]:
detector = LocationAwareTensorMahalanobisDetector(
    patches_per_image=normal.patches_per_image,
    iterations=3,
    covariance_shrinkage=0.2,
    location_fit_workers=1,
)

# One call fits on normal images, scores both sets, and saves diagnostics.
artifacts = detector.fit_and_save_diagnostics(
    normal,
    to_check,
    root / "diagnostics",
    grid_shape=(16, 16),
    formats=("npy", "json", "distribution", "heatmaps", "tiff"),
)
artifacts

In [ ]:
# Inspect the generated visual heatmaps directly in the notebook.
from IPython.display import Image as DisplayImage, display
for split in ("train", "test"):
    heatmap = root / "diagnostics" / split / "scores_heatmaps.png"
    if heatmap.exists():
        display(DisplayImage(filename=str(heatmap)))
    else:
        print(f"{heatmap} was not created; install matplotlib for PNG previews.")

The `train/` and `test/` directories contain the raw score arrays, JSON metadata, histograms, PNG heatmap previews, and one float32 TIFF score grid per image. These diagnostics are optional and do not change the detector or official evaluation.